<div class='bar_title'></div>

*Simulation for Decision Making (S4DM)*

# Exercise 2 — Condition Events, Preemption & Custom Signals

Summer Semester 26

Gunther Gust & Govind Rao <br>
Chair for Enterprise AI <br>
Data Driven Decisions Group <br>
Center for Artificial Intelligence and Data Science (CAIDAS)

<img src="images/d3.png" style="width:20%; float:left;" />

<img src="images/CAIDASlogo.png" style="width:20%; float:left;" />

## Learning Goals

After completing this exercise, you should be able to:

- Use **condition events** (`AnyOf` / `|`) to let a process wait for *either* of two events.
- Model a **`PreemptiveResource`** where high-priority requests interrupt low-priority ones.
- Use **`process.interrupt()`** to externally cancel a running process and handle `simpy.Interrupt`.
- Model continuous quantities with **`Container`** (`put` / `get` by amount).
- Create a **custom `simpy.Event`** and trigger it manually via `.succeed()` for cross-process signalling.

## Overview — four parts

| Part | Concept | Scenario |
|---|---|---|
| **A** | Condition events (`AnyOf` / `\|`) | Cinema with impatient moviegoers |
| **B** | Preemption & process interrupts | Machine shop with one repairman |
| **C** | Container & custom events | Gas station with low-fuel warning |
| **D** | Custom events as cross-process signals | Cinema sold-out fan-out |

In [1]:
import simpy
import random

---
## Part A — Condition Events (AnyOf / |)

A small cinema has **one ticket counter**. Moviegoers arrive and queue for a ticket, but they are **impatient**: each moviegoer has a personal *patience* drawn from `Uniform(1, 3)` minutes. If they don't reach the counter within that time, they **leave** (renege) without a ticket.

We follow the **modeling conventions** from the lecture:

- **Resource class** — `Cinema` (owns the ticket counter),
- **Entity class** — `Moviegoer` (with a `run()` method describing the customer's flow),
- **Generator function** — `moviegoer_generator()`,
- **Logger** — `EventLogger` to collect one row per moviegoer.

The `Cinema`, `EventLogger`, the generator, and the run block are already provided.
**Your job:** complete the `run()` method of the `Moviegoer` entity — this is the only new SimPy concept in this part (`|` / `AnyOf`).

In [2]:
# Parameters
RANDOM_SEED = 42
SERVICE_TIME = 1.0       # minutes per ticket
T_INTER = 0.5            # mean inter-arrival time (expovariate)
SIM_TIME = 60            # minutes

In [3]:
# Provided — Resource class (modeling convention)
class Cinema:
    def __init__(self, env, num_counters, service_time):
        self.env = env
        self.counter = simpy.Resource(env, capacity=num_counters)
        self.service_time = service_time

    def sell_ticket(self, name):
        yield self.env.timeout(self.service_time)

In [4]:
# Provided — EventLogger
import pandas as pd

class EventLogger:
    def __init__(self):
        self.logs = []

    def log(self, **kwargs):
        self.logs.append(kwargs)

    def get_df(self):
        return pd.DataFrame(self.logs)

### Task A — Complete `Moviegoer.run()`

Below is the `Moviegoer` entity class. Its `__init__` (including the `patience` attribute) and the surrounding scaffolding are given. **Complete the `run()` method** so that a moviegoer either gets served *or* gives up.

Inside the provided `with`-block, wait for **either** the counter request **or** the patience timeout, then branch on which one triggered: on success the moviegoer is served and the event is logged as `served=True`; on renege the event is logged as `served=False`. Both branches log the waiting time and the moviegoer's patience.

In [ ]:
class Moviegoer:
    def __init__(self,env, name, cinema,logger):
        self.env = env
        self.name = name
        self.cinema = cinema
        self.logger = logger
        self.patience = random.uniform(1, 3)
        
    def run(self):
         arrive = self.env.now
         with self.cinema.counter.request() as req:
            result = yield req | self.env.timeout(self.patience)
            wait = self.env.now - arrive

            if req in result:
                yield self.env.process(self.cinema.sell_ticket(self.name))
                self.logger(name = self.name, served = True, wait=wait, patience =self.patience)
            else:
                self.logger.log(name=self.name, served=False, wait=wait, patience=self.patience)

In [6]:
# SOLUTION
class Moviegoer:
    def __init__(self, env, name, cinema, logger):
        self.env = env
        self.name = name
        self.cinema = cinema
        self.logger = logger
        self.patience = random.uniform(1, 3)     # personal patience (minutes)

    def run(self):
        arrive = self.env.now

        with self.cinema.counter.request() as req:
            result = yield req | self.env.timeout(self.patience)
            wait = self.env.now - arrive

            if req in result:
                yield self.env.process(self.cinema.sell_ticket(self.name))
                self.logger.log(name=self.name, served=True,  wait=wait, patience=self.patience)
            else:
                self.logger.log(name=self.name, served=False, wait=wait, patience=self.patience)

In [7]:
# Provided — generator + run block (5-step pattern)
def moviegoer_generator(env, cinema, logger):
    i = 0
    while True:
        yield env.timeout(random.expovariate(1 / T_INTER))
        mg = Moviegoer(env, f"Moviegoer {i}", cinema, logger)
        env.process(mg.run())
        i += 1


random.seed(RANDOM_SEED)
env = simpy.Environment()
cinema = Cinema(env, num_counters=1, service_time=SERVICE_TIME)
logger = EventLogger()

env.process(moviegoer_generator(env, cinema, logger))
env.run(until=SIM_TIME)

df = logger.get_df()
print(f"Total moviegoers: {len(df)}")
print(f"Served:           {df['served'].sum()}")
print(f"Reneged:          {(~df['served']).sum()}")
df.head()

Total moviegoers: 101
Served:           56
Reneged:          45


,name,served,wait,patience
0,Moviegoer 0,True,0.000000,1.050022
1,Moviegoer 1,True,0.839188,1.446421
2,Moviegoer 2,True,1.172392,2.353399
3,Moviegoer 4,False,1.059594,1.059594
4,Moviegoer 6,False,1.397675,1.397675


### Task A.2 — What-if: two counters

In the cell below, set up and run a **new** simulation with **2 counters** (keep all other parameters unchanged). Compare the number of renegers to Task A.1 and add your interpretation as a comment.

In [8]:
# SOLUTION
random.seed(RANDOM_SEED)
env2 = simpy.Environment()
cinema2 = Cinema(env2, num_counters=2, service_time=SERVICE_TIME)  # two counters
logger2 = EventLogger()

env2.process(moviegoer_generator(env2, cinema2, logger2))
env2.run(until=SIM_TIME)

df2 = logger2.get_df()
print(f"Total moviegoers: {len(df2)}")
print(f"Served:           {df2['served'].sum()}")
print(f"Reneged:          {(~df2['served']).sum()}")

# With 2 counters the queue moves twice as fast, so far fewer moviegoers run out
# of patience before reaching a counter — renegers drop significantly.

Total moviegoers: 101
Served:           94
Reneged:          7


---
## Part B — Preemption & Process Interrupts

A workshop has **n identical machines** that produce parts. Each machine **breaks down** occasionally — its working process is interrupted. Repairs are done by a **single repairman** modelled as a `PreemptiveResource`. The repairman also has **low-priority "other jobs"** that get **preempted** whenever a machine needs fixing.

**Your tasks:**
1. Implement the `Machine` class: normal operation *and* break-down behaviour via `process.interrupt()`.
2. Implement the `Repairman` class with a `PreemptiveResource` and a low-priority `other_jobs` loop.
3. Run the simulation and print a log line each time the other job is preempted.

Credits: adapted from the [SimPy documentation](https://simpy.readthedocs.io/en/latest/examples/machine_shop.html).

In [9]:
# Parameters
MEAN_PART_TIME = 10.0          # avg. minutes to produce one part
STD_PART_TIME = 2.0
MEAN_TIME_TO_FAILURE = 300.0   # avg. minutes between breakdowns
FAILURE_RATE = 1 / MEAN_TIME_TO_FAILURE
REPAIR_TIME = 30.0             # minutes to repair a machine
OTHER_JOB_DURATION = 30.0      # duration of one repairman side job
NUM_MACHINES = 10
SIM_DURATION_WEEKS = 1 / 7     # simulate one day
SIM_DURATION_MIN = SIM_DURATION_WEEKS * 7 * 24 * 60

### Task B1 — `Machine` class

In [10]:
# SOLUTION
class Machine:
    def __init__(self, env, name, repairman, mean_part_time, std_part_time):
        self.env = env
        self.name = name
        self.mean_part_time = mean_part_time
        self.std_part_time = std_part_time
        self.parts_made = 0
        self.is_broken = False
        self.run = env.process(self.working(repairman))
        env.process(self.breakdown())

    def working(self, repairman):
        while True:
            try:
                t = max(0, random.normalvariate(self.mean_part_time, self.std_part_time))
                yield self.env.timeout(t)
                self.parts_made += 1
            except simpy.Interrupt:
                self.is_broken = True
                print(f"{self.name} broke at {self.env.now:.1f} — waiting for repairman")
                with repairman.resource.request(priority=1) as req:
                    yield req
                    yield self.env.timeout(REPAIR_TIME)
                    print(f"{self.name} repaired at {self.env.now:.1f}")
                self.is_broken = False

    def breakdown(self):
        while True:
            yield self.env.timeout(random.expovariate(FAILURE_RATE))
            if not self.is_broken:
                self.run.interrupt()  # interrupt the working process

### Task B2 — `Repairman` class

In [11]:
# SOLUTION
class Repairman:
    def __init__(self, env, job_duration):
        self.env = env
        self.resource = simpy.PreemptiveResource(env, capacity=1)
        self.job_duration = job_duration
        self.preempt_count = 0

    def other_jobs(self):
        while True:
            with self.resource.request(priority=2) as req:
                yield req
                try:
                    yield self.env.timeout(self.job_duration)
                except simpy.Interrupt:
                    self.preempt_count += 1
                    print(f"Repairman's side job preempted at {self.env.now:.1f}.")

### Task B3 — Run the machine shop

Run the simulation and print both the parts-made per machine **and** the total number of preemptions.

In [12]:
# SOLUTION
random.seed(RANDOM_SEED)
env = simpy.Environment()
repairman = Repairman(env, OTHER_JOB_DURATION)

machines = []
for i in range(NUM_MACHINES):
    machines.append(Machine(env, f"Machine {i}", repairman, MEAN_PART_TIME, STD_PART_TIME))
env.process(repairman.other_jobs())

print("Running simulation...")
env.run(until=SIM_DURATION_MIN)
print("... done\n")

print(f"Machine shop results after {SIM_DURATION_WEEKS:.3f} weeks")
for m in machines:
    print(f"{m.name} made {m.parts_made} parts.")
print(f"\nSide job preempted {repairman.preempt_count} times.")

Running simulation...
Machine 5 broke at 2.0 — waiting for repairman
Repairman's side job preempted at 2.0.
Machine 7 broke at 30.5 — waiting for repairman
Machine 5 repaired at 32.0
Machine 7 repaired at 62.0
Machine 3 broke at 66.5 — waiting for repairman
Repairman's side job preempted at 66.5.
Machine 2 broke at 74.0 — waiting for repairman
Machine 4 broke at 74.7 — waiting for repairman
Machine 0 broke at 96.5 — waiting for repairman
Machine 3 repaired at 96.5
Machine 6 broke at 124.8 — waiting for repairman
Machine 2 repaired at 126.5
Machine 5 broke at 144.7 — waiting for repairman
Machine 3 broke at 152.3 — waiting for repairman
Machine 4 repaired at 156.5
Machine 0 repaired at 186.5
Machine 6 repaired at 216.5
Machine 6 broke at 234.0 — waiting for repairman
Machine 7 broke at 238.5 — waiting for repairman
Machine 4 broke at 239.7 — waiting for repairman
Machine 5 repaired at 246.5
Machine 2 broke at 256.1 — waiting for repairman
Machine 5 broke at 257.7 — waiting for repairman

---
## Part C — Container & Custom Events

A gas station has **one underground tank** (`Container`) with capacity 1000 L. Cars arrive, refuel a random amount, and leave. When the tank level drops below a threshold, a **custom `simpy.Event`** fires and wakes a **tank-truck** process that refills the tank.

**Your tasks:**
1. Build the `GasStation` resource class.
2. Build the `Car` entity class.

### Task C1 — `GasStation` resource class

Complete the `GasStation` class. The class structure and all familiar parts are provided — fill in only the new SimPy primitives: the `Container`, the custom `Event`, and the signalling logic (`succeed()`, waiting on an event, `put()`, re-arming).

In [ ]:
# SOLUTION
class GasStation:
    def __init__(self, env, logger):
        self.env = env
        self.logger = logger
        self.tank = simpy.Container(env, capacity=TANK_CAPACITY, init=INITIAL_LEVEL)
        self.low_fuel = simpy.Event(env)
        env.process(self.fuel_monitor())
        env.process(self.call_tank_truck())

    def fuel_monitor(self):
        while True:
            yield self.env.timeout(1)
            if self.tank.level < THRESHOLD and not self.low_fuel.triggered:
                self.low_fuel.succeed()

    def call_tank_truck(self):
        while True:
            yield self.low_fuel
            self.logger.log(event='truck_dispatched', time=self.env.now,
                            level=self.tank.level)
            yield self.env.timeout(TRUCK_TIME)
            amount = self.tank.capacity - self.tank.level
            yield self.tank.put(amount)
            self.logger.log(event='tank_refilled', time=self.env.now,
                            amount=amount)
            self.low_fuel = simpy.Event(self.env)  # re-arm

### Task C2 — `Car` entity class

Complete the `Car` entity. The structure is identical to `Moviegoer` from Part A — fill in only the new Container operation (`get`) and the pumping time.

In [14]:
# SOLUTION
class Car:
    def __init__(self, env, name, station, logger):
        self.env = env
        self.name = name
        self.station = station
        self.logger = logger
        self.needed = random.randint(MIN_CAR_REFUEL, MAX_CAR_REFUEL)

    def run(self):
        arrive = self.env.now
        yield self.station.tank.get(self.needed)
        wait = self.env.now - arrive
        yield self.env.timeout(self.needed / REFUEL_RATE)
        self.logger.log(event='car_served', name=self.name,
                        arrive=arrive, needed=self.needed, wait=wait)

In [15]:
# Provided — parameters, generator, and run block
TANK_CAPACITY = 1000
INITIAL_LEVEL = 200
THRESHOLD = 150
TRUCK_TIME = 10
REFUEL_RATE = 2
MIN_CAR_REFUEL = 20
MAX_CAR_REFUEL = 60
T_INTER_CAR = 5
SIM_TIME_GAS = 120


def car_generator(env, station, logger):
    i = 0
    while True:
        yield env.timeout(random.expovariate(1 / T_INTER_CAR))
        car = Car(env, f"Car {i}", station, logger)
        env.process(car.run())
        i += 1


random.seed(RANDOM_SEED)
env = simpy.Environment()
logger = EventLogger()
station = GasStation(env, logger)

env.process(car_generator(env, station, logger))
env.run(until=SIM_TIME_GAS)

df_c = logger.get_df()
cars   = df_c[df_c['event'] == 'car_served']
trucks = df_c[df_c['event'].isin(['truck_dispatched', 'tank_refilled'])]
print(f"Cars served: {len(cars)},  mean wait: {cars['wait'].mean():.1f} min")
print(f"Truck events ({len(trucks)} rows):")
print(trucks[['event', 'time', 'level', 'amount']].to_string(index=False))

Cars served: 25,  mean wait: 0.2 min
Truck events (3 rows):
           event  time  level  amount
truck_dispatched  12.0  144.0     NaN
   tank_refilled  22.0    NaN   983.0
truck_dispatched 119.0  146.0     NaN


---
## Part D — Custom Events as Cross-Process Signals

Go back to the cinema from Part A. The cinema now has a **limited number of seats**. Each sold ticket reduces the remaining capacity. When the last seat is taken, a **`sold_out`** custom event fires — and every moviegoer still waiting in the counter queue should be kicked out immediately.

This combines what you practised in Part A (condition events) with the custom-event signalling from Part C.

**What changes compared to Part A:**
- `Cinema` gains a seat counter and a `sold_out` event (a `simpy.Event`).
- `sell_ticket()` fires `sold_out.succeed()` when the last seat is sold.
- `Moviegoer.run()` now waits for *three* events at once: the counter request, the patience timeout, **and** the sold-out signal.

Everything else (generator, run block, logger) is reused unchanged from Part A.

In [16]:
MAX_SEATS = 50   # cinema capacity (keep other parameters from Part A)

### Task D — Extend `Cinema` and `Moviegoer.run()` for sold-out

**`Cinema` changes:**
- Add `max_seats` and `seats_sold = 0` attributes.
- Add a `sold_out` attribute — a `simpy.Event` that starts un-triggered.
- In `sell_ticket()`, after the timeout, increment `seats_sold`. When `seats_sold` reaches `max_seats`, fire the `sold_out` event.

**`Moviegoer.run()` changes:**
- Inside the `with`-block, extend the yield to wait for three events: counter request, patience timeout, and the `sold_out` signal.
- Add a third branch: if the cinema is sold out, log the moviegoer as `served=False` with `reason='sold_out'`.

In [17]:
# SOLUTION
class Cinema:
    def __init__(self, env, num_counters, service_time, max_seats):
        self.env = env
        self.counter = simpy.Resource(env, capacity=num_counters)
        self.service_time = service_time
        self.max_seats = max_seats
        self.seats_sold = 0
        self.sold_out = simpy.Event(env)

    def sell_ticket(self, name):
        yield self.env.timeout(self.service_time)
        self.seats_sold += 1
        if self.seats_sold >= self.max_seats:
            print(f"SOLD OUT at {self.env.now:.2f} — {self.seats_sold} seats sold.")
            self.sold_out.succeed()


class Moviegoer:
    def __init__(self, env, name, cinema, logger):
        self.env = env
        self.name = name
        self.cinema = cinema
        self.logger = logger
        self.patience = random.uniform(1, 3)

    def run(self):
        arrive = self.env.now

        with self.cinema.counter.request() as req:
            result = yield req | self.env.timeout(self.patience) | self.cinema.sold_out
            wait = self.env.now - arrive

            if self.cinema.sold_out in result:
                self.logger.log(name=self.name, served=False, wait=wait,
                                patience=self.patience, reason='sold_out')
            elif req in result:
                yield self.env.process(self.cinema.sell_ticket(self.name))
                self.logger.log(name=self.name, served=True, wait=wait,
                                patience=self.patience, reason='served')
            else:
                self.logger.log(name=self.name, served=False, wait=wait,
                                patience=self.patience, reason='reneged')


# Reuse generator and run block from Part A
random.seed(RANDOM_SEED)
env = simpy.Environment()
cinema = Cinema(env, num_counters=1, service_time=SERVICE_TIME, max_seats=MAX_SEATS)
logger_d = EventLogger()

env.process(moviegoer_generator(env, cinema, logger_d))
env.run(until=SIM_TIME)

df_d = logger_d.get_df()
print(df_d['reason'].value_counts().to_string())

SOLD OUT at 52.27 — 50 seats sold.
reason
served      50
reneged     42
sold_out    11


---
## Short Questions

*The following questions are comparable to exam questions in format and expected answer length. Note that content-wise they are on the harder end — use them to stress-test your understanding.*

**Q1** — Given this system description: *"A printer queue accepts jobs. Each job takes 2–5 minutes to print. If a job waits more than 10 minutes it is cancelled."* Sketch the full class structure (Resource, Entity, generator) with the key attributes and `yield` statements. Sketch the key methods — you do not need to write fully executable code.

In [18]:
# SOLUTION
class PrinterQueue:
    def __init__(self, env):
        self.env = env
        self.printer = simpy.Resource(env, capacity=1)

class PrintJob:
    def __init__(self, env, name, queue):
        self.env = env
        self.name = name
        self.queue = queue
        self.patience = 10

    def run(self):
        with self.queue.printer.request() as req:
            result = yield req | self.env.timeout(self.patience)
            if req in result:
                yield self.env.timeout(random.uniform(2, 5))
                # log: printed
            else:
                pass  # log: cancelled

def job_generator(env, queue):
    i = 0
    while True:
        yield env.timeout(...)   # inter-arrival time
        job = PrintJob(env, f"Job {i}", queue)
        env.process(job.run())
        i += 1

**Q2** — A colleague writes this entity `run()` method:

```python
def run(self):
    req = self.resource.request()
    yield req
    yield self.env.timeout(5)
    self.resource.release(req)
```

An experienced modeler replaces it with a `with` block. What problem does the original code have, and what does the `with` block fix?

**Answer:** If the process is interrupted (e.g. `process.interrupt()`) or an exception occurs after `yield req` but before `self.resource.release(req)`, the release is never called. The resource slot remains occupied permanently, starving all future requesters. The `with` block guarantees release on any exit path — normal, exception, or interrupt.

**Q3** — Model a warehouse where a `Container` belt is filled by suppliers and drained by packers. When the belt drops below 10 items, a `restock` event fires and wakes a supplier. After restocking, the event must be re-armed. Write the supplier class with `__init__`, `monitor()`, and `restock_belt()` methods. Sketch the key methods — you do not need to write fully executable code.

In [19]:
# SOLUTION
class Supplier:
    def __init__(self, env, belt):
        self.env = env
        self.belt = belt          # simpy.Container
        self.restock = simpy.Event(env)
        env.process(self.monitor())
        env.process(self.restock_belt())

    def monitor(self):
        while True:
            yield self.env.timeout(1)
            if self.belt.level < 10 and not self.restock.triggered:
                self.restock.succeed()

    def restock_belt(self):
        while True:
            yield self.restock
            amount = self.belt.capacity - self.belt.level
            yield self.belt.put(amount)
            self.restock = simpy.Event(self.env)  # re-arm

---
## Wrap-up — reflect

**1. Part A:** When the patience timeout fires first, `req` is still pending in the queue. Does SimPy automatically cancel it when the `with`-block exits — and what would happen if it did not?

**Answer:** Yes — the `with`-block calls `req.__exit__()`, which invokes `resource.release(req)` even if the request was never granted. SimPy removes the pending request from the queue. If it did *not* do this, the ghost request would sit at the front of the queue indefinitely, blocking every moviegoer behind it: the resource would appear busy even though no process is actually using it, and the simulation would stall.

---

**2. Part B:** What would change if we used a plain `Resource` instead of a `PreemptiveResource` for the repairman? Would repairs still happen?

**Answer:** Repairs would still happen, but urgent jobs could no longer interrupt ongoing ones. With a plain `Resource`, once the repairman starts on a job he finishes it before picking up the next one — regardless of priority. `PreemptiveResource` allows a higher-priority request to seize the resource mid-process (triggering an `Interrupt` on the current holder), so urgent machines get attended to immediately. Dropping to a plain `Resource` turns the shop into a simple FIFO queue with no preemption.

---

**3. Part C:** Why is it important to re-arm the `low_fuel` event after each refill? What would happen if we didn't?

**Answer:** A `simpy.Event` can only be triggered once — once it fires, it stays in the "processed" state and will never fire again. If we reused the same event object after a refill, the next time the tank drops below the threshold nothing would happen: the condition check would see an already-triggered event and either skip it or raise an error. Re-arming means replacing it with a fresh `simpy.Event()` so the monitoring process can `yield` on it again for the next low-fuel episode.